#Extract

In [1]:
import os
import requests
import json
from datetime import datetime
from io import StringIO
import pandas as pd
from google.cloud import storage
import logging

In [2]:
logging.basicConfig(level=logging.INFO)

In [3]:
API_KEY = "7c77e6d6a551915e3890b158"
url = f"https://v6.exchangerate-api.com/v6/{API_KEY}/latest/USD"

In [4]:
response = requests.get(url)
data = response.json()

In [5]:
if response.status_code == 200 and data["result"] == "success":
  rates = data['conversion_rates']
  logging.info("USD to GBP:", rates["GBP"])
else:
  logging.error("API Error:", data)



In [6]:
if response.status_code != 200 or data['result'] != "success":
  raise RuntimeError(f"API Error: {data}")

In [7]:
date = data['time_last_update_utc']
dt = datetime.strptime(date, "%a, %d %b %Y %H:%M:%S %z")
formatted_date = dt.strftime("%Y-%m-%d")
logging.info(f"The date is {formatted_date}")

In [8]:
dir = f"data/raw/api_currency/{formatted_date}"
os.makedirs(dir, exist_ok=True)

In [9]:
path = os.path.join(dir, "response.json")
with open(path, "w", encoding="utf-8") as f:
  json.dump(data, f, indent=2)

In [10]:
cs = pd.read_csv("data/raw/clickstream.csv", chunksize=50000)
tr = pd.read_csv("data/raw/transactions.csv", chunksize=50000)

#Transform/ Load


In [11]:
def transform_tr(df):
  df.columns = df.columns.str.lower().str.strip()
  df.rename(columns={"txn_id":"transaction_id", "txn_time":"transaction_time"}, inplace=True)
  df["transaction_time"] = pd.to_datetime(df["transaction_time"], utc=True)
  df = df.drop_duplicates(subset=["transaction_id"])
  df["amount_in_usd"] = df.apply( lambda row: round(row["amount"]/rates[row["currency"]],2), axis = 1)
  return df


In [12]:
def transform_cs(df):
  df.columns = df.columns.str.lower().str.strip()
  df["click_time"] = pd.to_datetime(df["click_time"], utc=True)
  df = df.drop_duplicates(subset=["session_id"])
  return df

In [13]:
def write_to_gcs(df, name, part):
  file_name = f"{name}part{str(part)}.csv"
  path = f"{name}/ingest_date={formatted_date}/{file_name}"

  csv_buffer = StringIO()
  df.to_csv(csv_buffer, index=False)

  client = storage.Client()
  bucket = client.bucket("week_1_bucke")
  blob = bucket.blob(path)
  blob.upload_from_string(csv_buffer.getvalue(), content_type="text/csv")
  logging.info(f"Uploaded {df.shape[0]} rows to gs://week_1_bucke/{path}")


In [14]:
for i, chunk in enumerate(cs):
  transformed_cs = transform_cs(chunk)
  write_to_gcs(transformed_cs, "clickstream",i)

In [15]:
for i, chunk in enumerate(tr):
  transformed_tr = transform_tr(chunk)
  write_to_gcs(transformed_tr, "transactions",i)

In [16]:
logging.info("The datasets have been extracted, transformed and loaded")